In [1]:
import re
import pandas as pd
import copy
import numpy as np
import evaluate
from docx import Document
import os
import unittest
from datasets import load_dataset
import pickle
import matplotlib.pyplot as plt
from bleurt import score
class DataLoader:
    """
    Loader for benchmarking datasets to ensure universal formatting. To be used in conjunction with DyslexiaInjector.
    ...
    Attributes
    ----------
    path: str
        Path to csv, txt or docx file of the data. In the case of CSV there should only be 1 column
    data: list
        A list of striings
    dataset_name: str
        Name of the dataset that is used when saving the data
    ...
    Methods
    -------
    parse_txt(path)
        Parses a txt file and returns a list of strings
    fix_format(sentence)
        Fixes the formatting of a sentence
    save_as_txt(path)
        Saves the data as a txt file
    save_as_csv(path)
        Saves the data as a csv file
    save_as_docx(path)
        Saves the data as a docx file
    get_data()
        Returns the data
    create_deepcopy()
        Returns a deepcopy of the DataLoader instance
    get_name()
        Returns the dataset name
    get_number_of_sentences()
        Returns the number of sentences in the data
    get_number_of_words()
        Returns the number of words in the data
    get_number_of_letters()
        Returns the number of letters in the data
    edit_distance(reference_sentence, sentence)
        Returns the number of edits required to transform reference_sentence into sentence at word level
        edits include insertions, deletions and substitutions
        based on levenshtein distance
        also returns a dictionary of substitutions, insertions and deletions
    get_edit_distance(reference, manual_wer=False)
        Returns the number of edits required to transform data into reference at word level, substitutions, insertions and deletions the associated dictionaries
        and the WER (withouth alignment) if manual_wer is set to True
    get_individual_edit_distance(reference)
        Returns the number of edits required to transform data into reference at word level for each individual sentence
    combine_nested_dict(dict1, dict2)
        Combines two nested dictionaries
    combine_dicts(dict1, dict2)
        Combines two dictionaries
    get_bleue_score(reference)
        Returns bleu score of the data against a reference
    get_wer(reference)
        Returns the Word Error Rate (WER) of the data against a reference. With word alignment
    get_bert_score(reference)
        Returns the BERT Score similarity score of the data against a reference
    get_LaBSE(reference, model=None, tokenizer=None)
        Returns the LaBSE similarity score of the data against a reference which is a l2 norm between the reference and target sentences score.
        Score of 1 means the sentences are identical, closer to 0 means they are less similar semantically.
    ...

    Usage
    -------
    >>> from datasets import load_dataset
    >>> from DataLoader import DataLoader
    >>> dataset_wmt_enfr = load_dataset("wmt14",'fr-en', split='test')
    >>> to_translate = []
    >>> for i in range(len(dataset_wmt_enfr)):
    >>>     to_translate.append(dataset_wmt_enfr[i]['translation']['en'])
    >>> loader = DataLoader(data=to_translate, dataset_name="wmt14_enfr")
    >>> loader.save_as_txt("wmt14_enfr.txt")
    We can also use the text file to create a new DataLoader instance
    >>> loader2 = DataLoader(path="wmt14_enfr.txt", dataset_name="wmt14_enfr")
    """
    # Constructor
    def __init__(self, path=None, data=None, dataset_name=""):
        self.dataset_name = dataset_name
        if data is None and path is not None:
            #check path to see if file is txt or csv
            file_type = path.split(".")[-1]
            if file_type == "txt":
                self.data = self.parse_txt(path)
                self.data = [self.fix_format(sentence) for sentence in self.data]
            elif file_type == "csv":
                self.data = pd.read_csv(path, header=None)
                self.data = self.data[0].tolist()
                #fix any formatting issues
                self.data = [self.fix_format(sentence) for sentence in self.data]
            elif file_type == "docx":
                doc = Document(path)
                self.data = [self.fix_format(paragraph.text) for paragraph in doc.paragraphs]
            else:
                raise Exception("Invalid file type")
        elif data is not None:
            #check if data is a list or a df
            if isinstance(data, list):
                #format each sentence in data
                self.data = [self.fix_format(sentence) for sentence in data]
            else:
                raise Exception("Invalid data type, please pass in a list of sentences")
        else:
            raise Exception("Please pass in a path or data")

    def parse_txt(self, path):
        output = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                output.append(self.fix_format(line))
        return output
                
    def fix_format(self, sentence):
        #remove spacing before punctuation
        sentence = re.sub(r'\s([?.!,"](?:\s|$))', r'\1', sentence)
        #replace any double spaces with single space
        sentence = re.sub(r'\s+', ' ', sentence)
        #remove any leading or trailing spaces
        sentence = sentence.strip()
        #make all quotes (german and french) english double quotes
        sentence = re.sub(r'«|»|„|“', '"', sentence)
        #make all single quotes english single quotes
        sentence = re.sub(r'‘|’', "'", sentence)
        #make all french guillemets english double quotes
        sentence = re.sub(r'‹|›', '"', sentence)
        #if sentence begins and ends with quotes and there are only two, remove them
        if sentence[0] == '"' and sentence[-1] == '"' and sentence.count('"') == 2:
            sentence = sentence[1:-1]
        elif sentence[0] == "'" and sentence[-1] == "'" and sentence.count("'") == 2:
            sentence = sentence[1:-1]
        return sentence

    def save_as_txt(self, path):
        with open(path, "w", encoding="utf-8") as f:
            for sentence in self.data:
                f.write(f"{sentence}\n")
        print(f"Saved {self.dataset_name} to {path}")
        return
    
    def save_as_csv(self, path):
        df = pd.DataFrame(self.data)
        df.to_csv(path, index=False, header=False, encoding='utf-8')
        print(f"Saved {self.dataset_name} to {path}")
        return
    
    def save_as_docx(self, path):
        document = Document()
        for sentence in self.data:
            document.add_paragraph(sentence)
        document.save(path)
        print(f"Saved {self.dataset_name} to {path}")
        return

    def get_data(self):
        return self.data

    def create_deepcopy(self):
        return DataLoader(data=copy.deepcopy(self.data), dataset_name=self.dataset_name)
        
    def get_name(self):
        return self.dataset_name

    def get_number_of_sentences(self):
        return len(self.data)
    
    def get_number_of_words(self):
        return sum([len(sentence.split()) for sentence in self.data])
    
    def get_number_of_letters(self):
        #need to ensure we only count letters and not punctuation
        return sum([len(re.sub(r'[^\w\s]','',sentence)) for sentence in self.data])

    def edit_distance(reference_sentence, sentence):
        """
        Returns the number of edits required to transform reference_sentence into sentence at word level
        edits include insertions, deletions and substitutions
        based on levenshtein distance
        also returns a dictionary of substitutions, insertions and deletions
        """
        substitutions = 0
        insertions = 0
        deletions = 0
        substitution_dict = {}
        insertion_dict = {}
        deletion_dict = {}
        #remove punctuation and split into words
        sentence = re.sub(r'[^\w\s]','',sentence).lower().split()
        reference_sentence = re.sub(r'[^\w\s]','',reference_sentence).lower().split()
        #create matrix
        matrix = np.zeros((len(reference_sentence)+1,len(sentence)+1))
        #fill in first row and column
        for i in range(len(reference_sentence)+1):
            matrix[i][0] = i
        for j in range(len(sentence)+1):
            matrix[0][j] = j
        #fill in rest of matrix
        for i in range(1,len(reference_sentence)+1):
            for j in range(1,len(sentence)+1):
                if sentence[j-1] == reference_sentence[i-1]:
                    matrix[i][j] = matrix[i-1][j-1]
                else:
                    matrix[i][j] = min(matrix[i-1][j-1], matrix[i-1][j], matrix[i][j-1])+1
        #backtrack to find edits
        i = len(reference_sentence)
        j = len(sentence)
        while i > 0 and j > 0:
            if sentence[j-1] == reference_sentence[i-1]:
                i -= 1
                j -= 1
            else:
                if matrix[i][j] == matrix[i-1][j-1]+1:
                    substitutions += 1
                    if reference_sentence[i-1] not in substitution_dict:
                        substitution_dict[reference_sentence[i-1]] = {sentence[j-1]:1}
                    else:
                        if sentence[j-1] not in substitution_dict[reference_sentence[i-1]]:
                            substitution_dict[reference_sentence[i-1]][sentence[j-1]] = 1
                        else:
                            substitution_dict[reference_sentence[i-1]][sentence[j-1]] += 1
                    i -= 1
                    j -= 1
                elif matrix[i][j] == matrix[i-1][j]+1:
                    deletions += 1
                    if reference_sentence[i-1] not in deletion_dict:
                        deletion_dict[reference_sentence[i-1]] = 1
                    else:
                        deletion_dict[reference_sentence[i-1]] += 1
                    i -= 1
                elif matrix[i][j] == matrix[i][j-1]+1:
                    insertions += 1
                    if sentence[j-1] not in insertion_dict:
                        insertion_dict[sentence[j-1]] = 1
                    else:
                        insertion_dict[sentence[j-1]] += 1
                    j -= 1
        while i > 0:
            deletions += 1
            if reference_sentence[i-1] not in deletion_dict:
                deletion_dict[reference_sentence[i-1]] = 1
            else:
                deletion_dict[reference_sentence[i-1]] += 1
            i -= 1
        while j > 0:
            insertions += 1
            if sentence[j-1] not in insertion_dict:
                insertion_dict[sentence[j-1]] = 1
            else:
                insertion_dict[sentence[j-1]] += 1
            j -= 1
        distance = substitutions+insertions+deletions
        return substitutions, insertions, deletions, substitution_dict, insertion_dict, deletion_dict, distance
        
    def get_edit_distance(self, reference, manual_wer=False):
        """
        Returns the number of edits required to transform data into reference at word level, substitutions, insertions and deletions the associated dictionaries
        and the WER (withouth alignment) if manual_wer is set to True
        """
        if type(reference) == list:
            substitutions = 0
            insertions = 0
            deletions = 0
            all_sub = {}
            all_ins = {}
            all_del = {}
            distance = 0
            for i in range(len(self.data)):
                sub, ins, dele, substitution_dict, insertion_dict, deletion_dict, dist = DataLoader.edit_distance(reference[i], self.data[i], )
                all_sub = self.combine_nested_dict(all_sub, substitution_dict)
                all_ins = self.combine_dicts(all_ins, insertion_dict)
                all_del = self.combine_dicts(all_del, deletion_dict)
                substitutions += sub
                insertions += ins
                deletions += dele
                distance += dist
            if manual_wer:
                return substitutions, insertions, deletions, all_sub, all_ins, all_del, distance, distance/(sum([len(sentence.split()) for sentence in reference]))
            return substitutions, insertions, deletions, all_sub, all_ins, all_del, distance
        elif type(reference) == DataLoader:
            return self.get_edit_distance(reference.get_data(), manual_wer=manual_wer)
        else:
            raise Exception("Invalid reference type, please pass in a list or DataLoader instance")

    def get_individual_edit_distance(self, reference):
        """
        Returns the number of edits required to transform data into reference at word level for each individual sentence
        """
        if type(reference) == list:
            output = []
            for i in range(len(self.data)):
                sub, ins, dele, substitution_dict, insertion_dict, deletion_dict, distance = DataLoader.edit_distance(reference[i], self.data[i], )
                output.append((sub, ins, dele, substitution_dict, insertion_dict, deletion_dict, distance))
            return output
        elif type(reference) == DataLoader:
            return self.get_individual_edit_distance(reference.get_data())
        else:
            raise Exception("Invalid reference type, please pass in a list or DataLoader instance")       

    def combine_nested_dict(self, dict1, dict2):
        for key in dict2:
            if key not in dict1:
                dict1[key] = dict2[key]
            else:
                for key2 in dict2[key]:
                    if key2 not in dict1[key]:
                        dict1[key][key2] = dict2[key][key2]
                    else:
                        dict1[key][key2] += dict2[key][key2]
        return dict1
    
    def combine_dicts(self, dict1, dict2):
        for key in dict2:
            if key not in dict1:
                dict1[key] = dict2[key]
            else:
                dict1[key] += dict2[key]
        return dict1

    def get_bleue_score(self, reference):
        #returns bleu score of the data against a reference
        bleu = evaluate.load("bleu")
        if type(reference) == list:
            return bleu.compute(predictions=self.data, references=reference)
        elif type(reference) == DataLoader:
            return bleu.compute(predictions=self.data, references=reference.get_data())
        else:
            raise Exception("Invalid reference type, please pass in a list or DataLoader instance")

    def get_wer(self, reference):
        """
        Returns the Word Error Rate (WER) of the data against a reference. With word alignment
        """
        wer = evaluate.load("wer")
        if type(reference) == list:
            return wer.compute(predictions=self.data, references=reference)
        elif type(reference) == DataLoader:
            return wer.compute(predictions=self.data, references=reference.get_data())
        else:
            raise Exception("Invalid reference type, please pass in a list or DataLoader instance")


    def get_bert_score(self, reference, lang="fr"):
        """
        Returns the BERT Score similarity score of the data against a reference.
        """
        bert = evaluate.load("bertscore")
        if type(reference) == list:
            return bert.compute(predictions=self.data, references=reference, lang=lang)
        elif type(reference) == DataLoader:
            return bert.compute(predictions=self.data, references=reference.get_data(), lang=lang)
        else:
            raise Exception("Invalid reference type, please pass in a list or DataLoader instance")

    def get_LaBSE(self, reference, model=None, tokenizer=None):
        """
        Returns the LaBSE similarity score of the data against a reference which is a l2 norm between the reference and target sentences score.
        Score of 1 means the sentences are identical, closer to 0 means they are less similar semantically.
        """
        if model is None:
            model = BertModel.from_pretrained("setu4993/LaBSE")
        if tokenizer is None:
            tokenizer = BertTokenizerFast.from_pretrained("setu4993/LaBSE")
        if type(reference) == list:
            pass
        elif type(reference) == DataLoader:
            reference = reference.get_data()
        else:
            raise Exception("Invalid reference type, please pass in a list or DataLoader instance")
        target = self.data
        reference_inputs = tokenizer(reference, return_tensors="pt", padding=True).to("cuda")
        target_inputs = tokenizer(target, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            reference_outputs = model(**reference_inputs)
            target_outputs = model(**target_inputs)
        reference_embeddings = reference_outputs.pooler_output
        target_embeddings = target_outputs.pooler_output
        return self.similarity(reference_embeddings, target_embeddings)
    
    def get_bleurt(self, reference, scorer = None):
        """
        BLEURT-20 is required and can be downloaded via https://github.com/google-research/bleurt
        This is the most up to date version of BLEURT and is multilingual
        Returns the BLEURT similarity score of the data against a reference.
        """
        if scorer is None:
            try:
                scorer = score.BleurtScorer("BLEURT-20")
            except:
                raise Exception("BLEURT-20 not found")
        if type(reference) == list:
            scores = scorer.score(references = reference, candidates = self.data)
        elif type(reference) == DataLoader:
            scores = scorer.score(references = reference.get_data(), candidates = self.data)
        else:
            raise Exception("Invalid reference type, please pass in a list or DataLoader instance")
        return scores
    def get_COMET(self, reference, source):
        """
        Returns the COMET similarity score of the data against a reference and a source.
        Source is the original sentence and reference is the translation
        """
        comet = evaluate.load("comet")
        if type(reference) == list:
            return comet.compute(predictions=self.data, references=reference, sources=source)
        elif type(reference) == DataLoader:
            return comet.compute(predictions=self.data, references=reference.get_data(), sources=source.get_data())
        else:
            raise Exception("Invalid reference type, please pass in a list or DataLoader instance")

In [2]:
#need to loop through file directory
aws_data = []

temp = DataLoader(path="output_data\\reddit_text\\aws\\en.french_dys_data_raw.txt", dataset_name="aws_raw_dyslexia_translated")
aws_data.append(temp)

# azure_data = []

# temp = DataLoader(path="output_data\\reddit_text\\azure\\combined_raw_translated.txt", dataset_name="azure_raw_dyslexia_translated")
# azure_data.append(temp)


google_data = []

temp = DataLoader(path="output_data\\reddit_text\google\\translated_FR_dys_data_raw_google.docx", dataset_name="google_raw_dyslexia_translated")
google_data.append(temp)

gpt_data = []

temp = DataLoader(path="output_data\\reddit_text\gpt\\dys_FR_gpt_raw.txt", dataset_name="gpt_raw_dyslexia_translated")
gpt_data.append(temp)



# from datasets import load_dataset
# dataset_wmt_enfr = load_dataset("wmt14",'fr-en', split='test')
to_translate_wmt14_en = []
reference_wmt14_fr = []

# for i in range(len(dataset_wmt_enfr)):
#     to_translate_wmt14_en.append(dataset_wmt_enfr[i]['translation']['en'])
#     reference_wmt14_fr.append(dataset_wmt_enfr[i]['translation']['fr'])

aws_reference_corpus_fr = DataLoader(path="output_data\\reddit_text\\aws\\en.french_dys_data_corrected.txt", dataset_name="aws_fr_reference")
# azure_reference_corpus_fr = DataLoader(path="output_data\\reddit_text\\azure\\combined_corrected_translated.txt", dataset_name="azure_fr_reference")
google_reference_corpus_fr = DataLoader(path="output_data\\reddit_text\google\\translated_FR_dys_data_corrected_google.docx", dataset_name="google_fr_reference")
gpt_reference_corpus_fr = DataLoader(path="output_data\\reddit_text\gpt\\dys_FR_gpt_corrected.txt", dataset_name="gpt_fr_reference")
reference_corpus_en = DataLoader(path="r_Dyslexia_Text\default_files\corrected\\french_dys_data_corrected.txt", dataset_name="french_fr_corrected")


In [3]:
aws_bleu_scores = []
for data in aws_data:
    print(data.get_name())
    aws_bleu_scores.append(data.get_bleue_score(aws_reference_corpus_fr))
print(f"AWS BLEU scores: {aws_bleu_scores}")

google_bleu_scores = []
for data in google_data:
    print(data.get_name())
    google_bleu_scores.append(data.get_bleue_score(google_reference_corpus_fr))
print(f"Google BLEU scores: {google_bleu_scores}")

# azure_bleu_scores = []
# for data in azure_data:
#     print(data.get_name())
#     azure_bleu_scores.append(data.get_bleue_score(azure_reference_corpus_fr))
# print(f"Azure BLEU scores: {azure_bleu_scores}")

gpt_bleu_scores = []
for data in gpt_data:
    print(data.get_name())
    gpt_bleu_scores.append(data.get_bleue_score(gpt_reference_corpus_fr))
print(f"GPT BLEU scores: {gpt_bleu_scores}")

#same of above but for WER
aws_wer_scores = []
for data in aws_data:
    print(data.get_name())
    aws_wer_scores.append(data.get_wer(aws_reference_corpus_fr))
print(f"AWS WER scores: {aws_wer_scores}")

google_wer_scores = []
for data in google_data:
    print(data.get_name())
    google_wer_scores.append(data.get_wer(google_reference_corpus_fr))
print(f"Google WER scores: {google_wer_scores}")

# azure_wer_scores = []
# for data in azure_data:
#     print(data.get_name())
#     azure_wer_scores.append(data.get_wer(azure_reference_corpus_fr))
# print(f"Azure WER scores: {azure_wer_scores}")

gpt_wer_scores = []
for data in gpt_data:
    print(data.get_name())
    gpt_wer_scores.append(data.get_wer(gpt_reference_corpus_fr))
print(f"GPT WER scores: {gpt_wer_scores}")


aws_raw_dyslexia_translated
AWS BLEU scores: [{'bleu': 0.6731736009380933, 'precisions': [0.8254709637241164, 0.7085992610010077, 0.626253418413856, 0.5620015948963317], 'brevity_penalty': 0.9993774319267295, 'length_ratio': 0.9993776256418235, 'translation_length': 6423, 'reference_length': 6427}]
google_raw_dyslexia_translated
Google BLEU scores: [{'bleu': 0.7327631473167087, 'precisions': [0.8616183879093199, 0.7652558218595954, 0.6930181012190617, 0.6309403437815976], 'brevity_penalty': 1.0, 'length_ratio': 1.0085741505239758, 'translation_length': 6352, 'reference_length': 6298}]
gpt_raw_dyslexia_translated
GPT BLEU scores: [{'bleu': 0.6197789496354891, 'precisions': [0.7909362975630612, 0.6624923640806353, 0.5703009373458313, 0.4937655860349127], 'brevity_penalty': 1.0, 'length_ratio': 1.0384786147698684, 'translation_length': 7017, 'reference_length': 6757}]
aws_raw_dyslexia_translated
AWS WER scores: [0.2431254191817572]
google_raw_dyslexia_translated
Google WER scores: [0.1954

In [4]:
aws_COMET_scores = []
for data in aws_data:
    print(data.get_name())
    aws_COMET_scores.append(data.get_COMET(aws_reference_corpus_fr, reference_corpus_en))
print(f"AWS COMET scores: {aws_COMET_scores}")


google_COMET_scores = []
for data in google_data:
    print(data.get_name())
    google_COMET_scores.append(data.get_COMET(google_reference_corpus_fr, reference_corpus_en))
print(f"Google COMET scores: {google_COMET_scores}")


# azure_COMET_scores = []
# for data in azure_data:
#     print(data.get_name())
#     azure_COMET_scores.append(data.get_COMET(azure_reference_corpus_fr, reference_corpus_en))
# print(f"Azure COMET scores: {azure_COMET_scores}")

gpt_COMET_scores = []
for data in gpt_data:
    print(data.get_name())
    gpt_COMET_scores.append(data.get_COMET(gpt_reference_corpus_fr, reference_corpus_en))
print(f"GPT COMET scores: {gpt_COMET_scores}")

# #load COMET scores from pkl file

# aws_COMET_scores = pickle.load(open("COMET_scores/reddit_aws_COMET_scores.pkl", "rb"))

# google_COMET_scores = pickle.load(open("COMET_scores/reddit_google_COMET_scores.pkl", "rb"))

# azure_COMET_scores = pickle.load(open("COMET_scores/reddit_azure_COMET_scores.pkl", "rb"))

# gpt_COMET_scores = pickle.load(open("COMET_scores/reddit_gpt_COMET_scores.pkl", "rb"))


aws_raw_dyslexia_translated


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.4.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint C:\Users\User\.cache\huggingface\hub\models--Unbabel--wmt22-comet-da\snapshots\f49d328952c3470eff6bb6f545d62bfdb6e66304\checkpoints\model.ckpt`
Encoder model frozen.
c:\Users\User\anaconda3\envs\dyslexia\lib\site-packages\pytorch_lightning\core\saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3080') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_pr

AWS COMET scores: [{'mean_score': 0.8516327154153446, 'scores': [0.871191680431366, 0.6656854152679443, 0.9368025660514832, 0.8274348974227905, 0.8793103098869324, 0.9019964337348938, 0.93490070104599, 0.46805885434150696, 0.9027196764945984, 0.8122637271881104, 0.6745917201042175, 0.6176436543464661, 0.8458210229873657, 0.9350263476371765, 0.8298903107643127, 0.7740122079849243, 0.8026366829872131, 0.8977043032646179, 0.7530957460403442, 0.862684965133667, 0.41621002554893494, 0.5659367442131042, 0.49306532740592957, 0.968776524066925, 0.9490488171577454, 0.9720262885093689, 0.9778221249580383, 0.8959618806838989, 0.6045680642127991, 0.8708842992782593, 0.8188366293907166, 0.6458074450492859, 0.9085790514945984, 0.9072678685188293, 0.6727640628814697, 0.8822398781776428, 0.8664977550506592, 0.6499732732772827, 0.7225477695465088, 0.7461910843849182, 0.9811413884162903, 0.9013960957527161, 0.7955134510993958, 0.7542259097099304, 0.8655432462692261, 0.6821818947792053, 0.727927267551422

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.4.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint C:\Users\User\.cache\huggingface\hub\models--Unbabel--wmt22-comet-da\snapshots\f49d328952c3470eff6bb6f545d62bfdb6e66304\checkpoints\model.ckpt`
Encoder model frozen.
c:\Users\User\anaconda3\envs\dyslexia\lib\site-packages\pytorch_lightning\core\saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Google COMET scores: [{'mean_score': 0.8916709340457469, 'scores': [0.7998387217521667, 0.8839650750160217, 0.9357887506484985, 0.9209657311439514, 0.8833618760108948, 0.8815311789512634, 0.8973196148872375, 0.6962924599647522, 0.9213243126869202, 0.8646629452705383, 0.8907141089439392, 0.7897875308990479, 0.6960165500640869, 0.9499619603157043, 0.8793855309486389, 0.809874415397644, 0.8395425081253052, 0.9028170108795166, 0.9506711363792419, 0.8448107838630676, 0.43545669317245483, 0.6087835431098938, 0.47274911403656006, 0.9687766432762146, 0.9219177961349487, 0.9720264077186584, 0.9363741874694824, 0.8796518445014954, 0.8219318985939026, 0.973848819732666, 0.8377353549003601, 0.5909506678581238, 0.9021771550178528, 0.9038613438606262, 0.6797645092010498, 0.9727306962013245, 0.8683778047561646, 0.9351602792739868, 0.8818541169166565, 0.8332502245903015, 0.9811413884162903, 0.9043565988540649, 0.8832594752311707, 0.9549238085746765, 0.9298078417778015, 0.7047554850578308, 0.9802010655

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.4.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint C:\Users\User\.cache\huggingface\hub\models--Unbabel--wmt22-comet-da\snapshots\f49d328952c3470eff6bb6f545d62bfdb6e66304\checkpoints\model.ckpt`
Encoder model frozen.
c:\Users\User\anaconda3\envs\dyslexia\lib\site-packages\pytorch_lightning\core\saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


GPT COMET scores: [{'mean_score': 0.8849780785122405, 'scores': [0.8795958757400513, 0.9377999901771545, 0.9346373677253723, 0.8819471001625061, 0.917046308517456, 0.8663614392280579, 0.9333521723747253, 0.982173502445221, 0.9186710119247437, 0.9282893538475037, 0.7999973297119141, 0.910700261592865, 0.9433432817459106, 0.9418061971664429, 0.8733972311019897, 0.8541809916496277, 0.8753931522369385, 0.7423610091209412, 0.8768755793571472, 0.9383211731910706, 0.4248533248901367, 0.7229971885681152, 0.6094397902488708, 0.8919320702552795, 0.8889365196228027, 0.9720264077186584, 0.8871533274650574, 0.8969746828079224, 0.9005922079086304, 0.7482423782348633, 0.8836371898651123, 0.5304260849952698, 0.843347430229187, 0.8872361779212952, 0.7853266596794128, 0.9056434631347656, 0.8842657208442688, 0.8066009283065796, 0.9321274161338806, 0.8716483116149902, 0.9811413288116455, 0.9280937314033508, 0.9169369339942932, 0.9801909327507019, 0.9110456705093384, 0.7643416523933411, 0.9259290099143982,

In [6]:
reddit_COMET = [aws_COMET_scores[0]['mean_score'], google_COMET_scores[0]['mean_score'], gpt_COMET_scores[0]['mean_score']]
df_COMET = pd.DataFrame([reddit_COMET], columns=['aws', 'google', 'gpt'])
df_COMET.index = ['French Dyslexia Text']
df_COMET.to_csv("For_graphs/french_dys_COMET_scores.csv")

In [7]:
aws_COMET_scores[0]['mean_score']

0.8516327154153446

In [8]:
def calculate_SE(data):
    data = [x for x in data]
    stdv = np.std(data)
    n = len(data)
    se = stdv/np.sqrt(n)
    return se

In [9]:
reddit_COMET_se = [calculate_SE(aws_COMET_scores[0]['scores']), calculate_SE(google_COMET_scores[0]['scores']), calculate_SE(gpt_COMET_scores[0]['scores'])]
df_COMET_se = pd.DataFrame([reddit_COMET_se], columns=['aws', 'google', 'gpt'])
df_COMET_se.index = ['French Dyslexia Text']
df_COMET_se.to_csv("For_graphs/french_dys_COMET_scores_standard_error.csv")

In [10]:
#save COMET scores to pickle files
import pickle
with open("COMET_scores/french_dys_aws_COMET_scores.pkl", "wb") as f:
    pickle.dump(aws_COMET_scores, f)

with open("COMET_scores/french_dys_google_COMET_scores.pkl", "wb") as f:
    pickle.dump(google_COMET_scores, f)

# with open("COMET_scores/v2_reddit_azure_COMET_scores.pkl", "wb") as f:
#     pickle.dump(azure_COMET_scores, f)

with open("COMET_scores/french_dys_gpt_COMET_scores.pkl", "wb") as f:
    pickle.dump(gpt_COMET_scores, f)


In [12]:
scorer = score.BleurtScorer("BLEURT-20")
aws_BLEURT_scores = []
for data in aws_data:
    print(data.get_name())
    aws_BLEURT_scores.append(data.get_bleurt(aws_reference_corpus_fr, scorer = scorer))
print(f"AWS BLEURT scores: {aws_BLEURT_scores}")


google_BLEURT_scores = []
for data in google_data:
    print(data.get_name())
    google_BLEURT_scores.append(data.get_bleurt(google_reference_corpus_fr, scorer = scorer))
print(f"Google BLEURT scores: {google_BLEURT_scores}")


# azure_BLEURT_scores = []
# for data in azure_data:
#     print(data.get_name())
#     azure_BLEURT_scores.append(data.get_bleurt(azure_reference_corpus_fr, scorer = scorer))
# print(f"Azure BLEURT scores: {azure_BLEURT_scores}")

gpt_BLEURT_scores = []
for data in gpt_data:
    print(data.get_name())
    gpt_BLEURT_scores.append(data.get_bleurt(gpt_reference_corpus_fr, scorer = scorer))
print(f"GPT BLEURT scores: {gpt_BLEURT_scores}")

# aws_BLEURT_scores = pickle.load(open("BLEURT_scores/reddit_aws_bleuRT_scores.pkl", "rb"))

# google_BLEURT_scores = pickle.load(open("BLEURT_scores/reddit_google_bleuRT_scores.pkl", "rb"))

# azure_BLEURT_scores = pickle.load(open("BLEURT_scores/reddit_azure_bleuRT_scores.pkl", "rb"))

# gpt_BLEURT_scores = pickle.load(open("BLEURT_scores/reddit_gpt_bleuRT_scores.pkl", "rb"))



INFO:tensorflow:Reading checkpoint BLEURT-20.


Reading checkpoint BLEURT-20.


INFO:tensorflow:Config file found, reading.


Config file found, reading.


INFO:tensorflow:Will load checkpoint BLEURT-20


...

INFO:tensorflow:Loading model.


Loading model.
Fingerprint not found. Saved model loading will continue.


INFO:tensorflow:BLEURT initialized.


BLEURT initialized.


aws_raw_dyslexia_translated


In [13]:
from statistics import mean 
reddit_bleuRT = [mean(aws_BLEURT_scores[0]), mean(google_BLEURT_scores[0]), mean(gpt_BLEURT_scores[0])]
df_bleuRT = pd.DataFrame([reddit_bleuRT], columns=['aws', 'google', 'gpt'])
df_bleuRT.index = ['French Dyslexia Text']
df_bleuRT.to_csv("For_graphs/french_dys_bleuRT_scores.csv")

In [14]:
reddit_bleuRT_se = [calculate_SE(aws_BLEURT_scores[0]), calculate_SE(google_BLEURT_scores[0]), calculate_SE(gpt_BLEURT_scores[0])]
df_bleuRT_se = pd.DataFrame([reddit_bleuRT_se], columns=['aws', 'google', 'gpt'])
df_bleuRT_se.index = ['French Dyslexia Text']
df_bleuRT_se.to_csv("For_graphs/french_dys_bleuRT_scores_standard_error.csv")

In [15]:
#save BLEURT scores to pickle files
import pickle
with open("BLEURT_scores/french_dys_aws_bleuRT_scores.pkl", "wb") as f:
    pickle.dump(aws_BLEURT_scores, f)

with open("BLEURT_scores/french_dys_google_bleuRT_scores.pkl", "wb") as f:
    pickle.dump(google_BLEURT_scores, f)

# with open("BLEURT_scores/v2_reddit_azure_bleuRT_scores.pkl", "wb") as f:
#     pickle.dump(azure_BLEURT_scores, f)

with open("BLEURT_scores/french_dys_gpt_bleuRT_scores.pkl", "wb") as f:
    pickle.dump(gpt_BLEURT_scores, f)


In [16]:
aws_bert_scores = []
for data in aws_data:
    print(data.get_name())
    aws_bert_scores.append(data.get_bert_score(aws_reference_corpus_fr, lang="en"))
print(f"AWS bert scores: {aws_bert_scores}")


google_bert_scores = []
for data in google_data:
    print(data.get_name())
    google_bert_scores.append(data.get_bert_score(google_reference_corpus_fr, lang="en"))
print(f"Google bert scores: {google_bert_scores}")


# azure_bert_scores = []
# for data in azure_data:
#     print(data.get_name())
#     azure_bert_scores.append(data.get_bert_score(azure_reference_corpus_fr))
# print(f"Azure bert scores: {azure_bert_scores}")

gpt_bert_scores = []
for data in gpt_data:
    print(data.get_name())
    gpt_bert_scores.append(data.get_bert_score(gpt_reference_corpus_fr, lang="en"))
print(f"GPT bert scores: {gpt_bert_scores}")

# aws_bert_scores = pickle.load(open("bert_scores/reddit_aws_bert_scores.pkl", "rb"))

# google_bert_scores = pickle.load(open("bert_scores/reddit_google_bert_scores.pkl", "rb"))

# azure_bert_scores = pickle.load(open("bert_scores/reddit_azure_bert_scores.pkl", "rb"))

# gpt_bert_scores = pickle.load(open("bert_scores/reddit_gpt_bert_scores.pkl", "rb"))

v2_aws_raw_dyslexia_translated
AWS bert scores: [{'precision': [0.8199949264526367, 0.8125854730606079, 0.9999999403953552, 0.974548876285553, 0.9586833715438843, 0.8174769878387451, 1.000000238418579, 1.0, 0.9552832245826721, 0.9071828126907349, 0.9149666428565979, 0.9314751029014587, 0.9397435188293457, 0.9553378820419312, 0.8849670886993408, 0.9399235844612122, 0.9898887276649475], 'recall': [0.8761842846870422, 0.808025598526001, 0.9999999403953552, 0.9847974181175232, 0.9592299461364746, 0.8313922882080078, 1.000000238418579, 1.0, 0.9616386294364929, 0.9106796979904175, 0.9414270520210266, 0.9405884742736816, 0.9418092966079712, 0.9583092927932739, 0.872308611869812, 0.9514371752738953, 0.9898887276649475], 'f1': [0.8471589088439941, 0.8102990984916687, 0.9999999403953552, 0.9796463251113892, 0.9589565992355347, 0.8243759274482727, 1.000000238418579, 1.0, 0.958450436592102, 0.9089279174804688, 0.9280083179473877, 0.9360095858573914, 0.9407752752304077, 0.9568212628364563, 0.878592

In [17]:
from statistics import mean
reddit_bert_score = [mean(aws_bert_scores[0]['f1']), mean(google_bert_scores[0]['f1']) mean(gpt_bert_scores[0]['f1'])]
df_bertScore = pd.DataFrame([reddit_bert_score], columns=['aws', 'google', 'gpt'])
df_bertScore.index = ['French Dyslexia Text']
df_bertScore.to_csv("For_graphs/french_dys_bert_scores.csv")

In [18]:
reddit_bert_score_se = [calculate_SE(aws_bert_scores[0]['f1']), calculate_SE(google_bert_scores[0]['f1']) calculate_SE(gpt_bert_scores[0]['f1'])]
df_bert_score_se = pd.DataFrame([reddit_bleuRT_se], columns=['aws', 'google', 'gpt'])
df_bert_score_se.index = ['French Dyslexia Text']
df_bert_score_se.to_csv("For_graphs/french_dys_bert_scores_standard_error.csv")

In [19]:
#save bert scores to pickle files
import pickle
with open("bert_scores/french_dys_aws_bert_scores.pkl", "wb") as f:
    pickle.dump(aws_bert_scores, f)

with open("bert_scores/french_dys_google_bert_scores.pkl", "wb") as f:
    pickle.dump(google_bert_scores, f)

# with open("bert_scores/v2_reddit_azure_bert_scores.pkl", "wb") as f:
#     pickle.dump(azure_bert_scores, f)

with open("bert_scores/french_dys_gpt_bert_scores.pkl", "wb") as f:
    pickle.dump(gpt_bert_scores, f)
